## Датасет собран из базы данных переписи 1994 года и содержит данные о доходах.
### Информация о данных:
* age: continuous.
* workclass: Private, Self-emp-not-inc, Self-emp-inc, Federal-gov, Local-gov, State-gov, Without-pay, Never-worked.
* fnlwgt: continuous.
* education: Bachelors, Some-college, 11th, HS-grad, Prof-school, Assoc-acdm, Assoc-voc, 9th, 7th-8th, 12th, * Masters, 1st-4th, 10th, Doctorate, 5th-6th, Preschool.
* education-num: continuous.
* marital-status: Married-civ-spouse, Divorced, Never-married, Separated, Widowed, Married-spouse-absent, Married-AF-spouse.
* occupation: Tech-support, Craft-repair, Other-service, Sales, Exec-managerial, Prof-specialty, Handlers-cleaners, Machine-op-inspct, Adm-clerical, Farming-fishing, Transport-moving, Priv-house-serv, Protective-serv, Armed-Forces.
* relationship: Wife, Own-child, Husband, Not-in-family, Other-relative, Unmarried.
* race: White, Asian-Pac-Islander, Amer-Indian-Eskimo, Other, Black.
* sex: Female, Male.
* capital-gain: continuous.
* capital-loss: continuous.
* hours-per-week: continuous.
* native-country: United-States, Cambodia, England, Puerto-Rico, Canada, Germany, Outlying-US(Guam-USVI-etc), India, Japan, Greece, South, China, Cuba, Iran, Honduras, Philippines, Italy, Poland, Jamaica, Vietnam, Mexico, Portugal, Ireland, France, Dominican-Republic, Laos, Ecuador, Taiwan, Haiti, Columbia, Hungary, Guatemala, Nicaragua, Scotland, Thailand, Yugoslavia, El-Salvador, Trinadad&Tobago, Peru, Hong, Holand-Netherlands.
* salary: >50K,<=50K

## Проведите анализ данных при помощи Pandas выполнив поставленные задачи.
####

In [3]:
import pandas as pd

In [4]:
# загружаем датасет
data = pd.read_csv("/content/adult.data.csv")
data.head()

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


**1. Посчитайте, сколько мужчин и женщин (признак *sex*) представлено в этом датасете**

In [5]:
sex_count=data['sex'].value_counts()
sex_count

,count
sex,
Male,21790
Female,10771


**2. Каков средний возраст мужчин (признак age) по всему датасету?**

In [6]:
man_avg_age = data[data['sex']=='Male'].age.mean().round()
man_avg_age

np.float64(39.0)

**3. Какова доля граждан Соединенных Штатов (признак native-country)?**

In [49]:
usa_citizens = ((data['native-country']=='United-States').mean())*100

print(f'USA citizens: {usa_citizens:.2f}%')

USA citizens: 89.59%


**4-5. Рассчитайте среднее значение и среднеквадратичное отклонение возраста тех, кто получает более 50K в год (признак salary) и тех, кто получает менее 50K в год**

In [8]:
mean_std_values = data.groupby('salary')['age'].agg(['mean', 'std'])
mean_std_values

,mean,std
salary,,
<=50K,36.783738,14.020088
>50K,44.249841,10.519028


**6. Правда ли, что люди, которые получают больше 50k, имеют минимум высшее образование? (признак education – Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters или Doctorate)**

In [9]:
high_edu_unis=['Bachelors', 'Prof-school', 'Assoc-acdm', 'Assoc-voc', 'Masters', 'Doctorate']

high_salary=data[data['salary']=='>50K']

rusult=high_salary['education'].isin(high_edu_unis).all()
rusult

np.False_

**7. Выведите статистику возраста для каждой расы (признак race) и каждого пола. Используйте groupby и describe. Найдите таким образом максимальный возраст мужчин расы Asian-Pac-Islander.**

In [10]:
stat_age = data.groupby(['race', 'sex'])['age'].describe()

max_man = data[(data['race']=='Asian-Pac-Islander') &
                (data['sex']=='Male')]['age'].max()

**8. Среди кого больше доля зарабатывающих много (>50K): среди женатых или холостых мужчин (признак marital-status)? Женатыми считаем тех, у кого marital-status начинается с Married (Married-civ-spouse, Married-spouse-absent или Married-AF-spouse), остальных считаем холостыми.**

In [76]:
men = data[data['sex']=='Male']

married = men[men['marital-status'].str.startswith('Married')]
unmarried = men[~men['marital-status'].str.startswith('Married')]

married_share = ((married['salary'].eq('>50K'))*100).mean()
unmarried_share = (unmarried['salary'] == '>50K').mean() * 100

share = pd.Series({
    'Married': married_share,
    'Unmarried' : unmarried_share
})

def who_is_winner(share):
  winner = share.idxmax()
  diff = share.max()-share.min()
  return f'{winner} man earn more on {diff:.2f}%'

print(who_is_winner(share))

Married man earn more on 35.60%


**9. Какое максимальное число часов человек работает в неделю (признак hours-per-week)? Сколько людей работают такое количество часов и каков среди них процент зарабатывающих много?**

In [12]:
max_hours = data['hours-per-week'].max()

max_hour_workers = data.loc[data['hours-per-week'] == max_hours,
                           ['salary', 'hours-per-week']]

max_hour_count = max_hour_workers.shape[0]

rich_share_max_hours = (data.loc[
    data['hours-per-week']==max_hours, 'salary']
    .eq('>50K')
    .mean()*100)

print(
    f"{'Maximum number of hours:':35} {max_hours}\n"
    f"{'Count of max-hour workers:':35} {max_hour_count}\n"
    f"{'Share earning >50K:':35} {rich_share_max_hours:.2f}%"
)


Maximum number of hours:            99
Count of max-hour workers:          85
Share earning >50K:                 29.41%


**10. Посчитайте среднее время работы (hours-per-week) зарабатывающих мало и много (salary) для каждой страны (native-country).**

In [13]:
avg_salary = data.groupby(['native-country', 'salary'])['hours-per-week'].mean()
avg_salary

native-country  salary
?               <=50K     40.164760
                >50K      45.547945
Cambodia        <=50K     41.416667
                >50K      40.000000
Canada          <=50K     37.914634
                            ...    
United-States   >50K      45.505369
Vietnam         <=50K     37.193548
                >50K      39.200000
Yugoslavia      <=50K     41.600000
                >50K      49.500000
Name: hours-per-week, Length: 82, dtype: float64

**11.Сгруппируйте людей по возрастным группам young, adult, retiree, где:**

*   young соответствует 16-35 лет
*   adult - 35-70 лет
*   retiree - 70-100 лет

**Проставьте название соответсвтуещей группы для каждого человека в новой колонке AgeGroup**

In [43]:
data['AgeGroup'] = pd.cut(data['age'],
                            bins=[15, 34, 69, 100],
                            labels=['young', 'adult', 'retiree'])
data

,age,AgeGroup,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,adult,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,adult,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,adult,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,adult,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,young,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
32556,27,young,Private,257302,Assoc-acdm,12,Married-civ-spouse,Tech-support,Wife,White,Female,0,0,38,United-States,<=50K
32557,40,adult,Private,154374,HS-grad,9,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0,0,40,United-States,>50K
32558,58,adult,Private,151910,HS-grad,9,Widowed,Adm-clerical,Unmarried,White,Female,0,0,40,United-States,<=50K
32559,22,young,Private,201490,HS-grad,9,Never-married,Adm-clerical,Own-child,White,Male,0,0,20,United-States,<=50K


**12-13. Определите количество зарабатывающих >50K в каждой из возрастных групп (колонка AgeGroup), а также выведите название возрастной группы, в которой чаще зарабатывают больше 50К (>50K)**

In [ ]:
high_income = data['salary'].eq('>50K')

max_salary_count = high_income.groupby(data['AgeGroup']).sum()
max_group=max_salary_count.idxmax()

**14. Сгруппируйте людей по типу занятости (колонка occupation) и определите количество людей в каждой группе. После чего напишите функциюю фильтрации filter_func, которая будет возвращать только те группы, в которых средний возраст (колонка age) не больше 40 и в которых все работники отрабатывают более 5 часов в неделю (колонка hours-per-week)**

In [1]:
occupation = data.groupby('occupation')[['hours-per-week']].agg('count')

def filter_func(group):
    avg_age = group['age'].mean() <= 40
    total_time = (group['hours-per-week'] > 5).all()
    return avg_age and total_time

filtered_groups = data.groupby('occupation').filter(filter_func)

result = (filtered_groups.groupby('occupation')
                         .agg({'age': 'mean',
                               'hours-per-week': 'min'}))

NameError: name 'data' is not defined